In [ ]:
# ============================================================
# STEERING INFERENCE ALPHA GRID FOR EMOTION CLASSIFICATION
# ============================================================

import ast
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Imports completed")


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(r"/") # your path
MODEL_DIR = PROJECT_DIR / "models" / "Qwen2.5-Math-1_5B"
STEERING_PATH = PROJECT_DIR / "steering_vectors" / "steering_emotion_qwen2_5.pt"
TEST_PATH = PROJECT_DIR / "emotion_dataset" / "simplified" / "test.parquet"

OUTPUT_DIR = PROJECT_DIR / "inference_outputs"
OUTPUT_STEM = OUTPUT_DIR / "steering_alpha_grid_qwen2_5_goemotions_test"
XLSX_OUT_PATH = OUTPUT_STEM.with_suffix(".xlsx")
PARQUET_OUT_PATH = OUTPUT_STEM.with_suffix(".parquet")
METRICS_PARQUET_PATH = OUTPUT_STEM.with_name(OUTPUT_STEM.name + "_metrics").with_suffix(".parquet")
PER_CLASS_PARQUET_PATH = OUTPUT_STEM.with_name(OUTPUT_STEM.name + "_per_class").with_suffix(".parquet")
CONFUSION_PARQUET_PATH = OUTPUT_STEM.with_name(OUTPUT_STEM.name + "_confusion_matrix").with_suffix(".parquet")


# ============================================================
# CONFIG
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TORCH_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32
GEN_MAX_NEW_TOKENS = 16
N_RUNS = 1
MAX_ROWS = None

ALPHA_GRID = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5]

TEXT_COLUMN = "text"
LABEL_COLUMN = "labels"

GOEMOTIONS_LABELS = {
    0: "admiration", 1: "amusement", 2: "anger", 3: "annoyance",
    4: "approval", 5: "caring", 6: "confusion", 7: "curiosity",
    8: "desire", 9: "disappointment", 10: "disapproval", 11: "disgust",
    12: "embarrassment", 13: "excitement", 14: "fear", 15: "gratitude",
    16: "grief", 17: "joy", 18: "love", 19: "nervousness",
    20: "optimism", 21: "pride", 22: "realization", 23: "relief",
    24: "remorse", 25: "sadness", 26: "surprise", 27: "neutral",
}
CLASS_NAMES = list(GOEMOTIONS_LABELS.values())


# ============================================================
# DATA AND LABEL UTILS
# ============================================================

def load_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported input format: {suffix}. Use XLSX, XLS, or Parquet.")


def decode_true_label(value):
    # GoEmotions is multi-label. Stage 3 uses the first label as a single
    # target, matching the training and stage 1 inference notebooks.
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple)):
        if not value:
            return None
        value = value[0]
    if isinstance(value, str):
        value = value.strip()
        if value.startswith("[") or value.startswith("("):
            try:
                return decode_true_label(ast.literal_eval(value))
            except (ValueError, SyntaxError):
                pass
        if value.isdigit():
            return GOEMOTIONS_LABELS.get(int(value))
        return value.lower() or None
    if value is None or pd.isna(value):
        return None
    return GOEMOTIONS_LABELS.get(int(value))


def prepare_dataset(path: Path) -> pd.DataFrame:
    df = load_table(path)
    df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN]).reset_index(drop=True)
    df["true_label"] = df[LABEL_COLUMN].map(decode_true_label)
    return df.dropna(subset=["true_label"]).reset_index(drop=True)


def parse_predicted_label(text: str) -> str:
    normalized = re.sub(r"[^a-z_ -]+", " ", text.lower()).strip()
    normalized = re.sub(r"\s+", " ", normalized)
    if normalized in CLASS_NAMES:
        return normalized
    matches = []
    for label in CLASS_NAMES:
        for match in re.finditer(rf"(?<!\w){re.escape(label)}(?!\w)", normalized):
            matches.append((match.start(), label))
    return max(matches)[1] if matches else "unknown"


def excel_safe(value):
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple, dict)):
        return json.dumps(value, ensure_ascii=False)
    return value


# ============================================================
# PROMPT
# ============================================================

def make_prompt(text: str) -> str:
    labels = ", ".join(CLASS_NAMES)
    messages = [
        {
            "role": "system",
            "content": (
                "Classify the emotion expressed in the text. "
                f"Choose exactly one label from: {labels}. "
                "Return only the lowercase label and no explanation."
            ),
        },
        {"role": "user", "content": text},
    ]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    return f"{messages[0]['content']}\n\nTEXT:\n{text}\n\nLABEL:\n"


# ============================================================
# MODEL
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    dtype=TORCH_DTYPE,
    device_map={"": 0} if DEVICE.type == "cuda" else None,
)
model.eval()
torch.set_grad_enabled(False)

print(f"Model loaded from: {MODEL_DIR}")
print(f"Device: {DEVICE}")


# ============================================================
# STEERING
# ============================================================

class Steering(nn.Module):
    def __init__(self, model, alpha):
        super().__init__()
        self.layers = model.model.layers
        hidden_size = model.config.hidden_size
        self.vectors = nn.Parameter(
            torch.zeros(len(self.layers), hidden_size, device=DEVICE)
        )
        self.alpha = alpha
        self.handles = []

    def install(self):
        self.remove()

        def make_hook(layer_index):
            def hook(_, __, output):
                if isinstance(output, tuple):
                    output = output[0]
                return output + (self.alpha * self.vectors[layer_index]).to(output.dtype)
            return hook

        for layer_index, layer in enumerate(self.layers):
            handle = layer.mlp.down_proj.register_forward_hook(make_hook(layer_index))
            self.handles.append(handle)

    def remove(self):
        for handle in self.handles:
            try:
                handle.remove()
            except Exception:
                pass
        self.handles = []


steering = Steering(model, 1.0)
steering.install()

checkpoint = torch.load(STEERING_PATH, map_location=DEVICE)
steering.vectors.data.copy_(
    checkpoint["vectors"].to(DEVICE, dtype=steering.vectors.dtype)
)
loaded_alpha = checkpoint.get("alpha", 1.0)
if checkpoint.get("class_names"):
    CLASS_NAMES = list(checkpoint["class_names"])

print(f"Loaded steering vectors: {tuple(steering.vectors.shape)}")
print(f"Training alpha stored in checkpoint: {loaded_alpha}")


# ============================================================
# GENERATE
# ============================================================

@torch.no_grad()
def generate_label(text: str, alpha: float):
    steering.alpha = alpha
    prompt = make_prompt(text)
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    seq = model.generate(
        **enc,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_tokens = seq[0, enc["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    predicted_label = parse_predicted_label(raw_output)
    return raw_output, predicted_label


# ============================================================
# METRICS
# ============================================================

def calculate_metrics(frame: pd.DataFrame, prediction_column: str, mode: str, alpha: float):
    y_true = frame["true_label"].tolist()
    y_pred = frame[prediction_column].tolist()

    summary = {
        "alpha": alpha,
        "mode": mode,
        "macro_f1": f1_score(y_true, y_pred, labels=CLASS_NAMES, average="macro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, labels=CLASS_NAMES, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, labels=CLASS_NAMES, average="macro", zero_division=0),
        "unknown_predictions": int((frame[prediction_column] == "unknown").sum()),
        "dataset_size": len(frame),
    }

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=CLASS_NAMES, zero_division=0
    )
    per_class = pd.DataFrame({
        "alpha": alpha,
        "mode": mode,
        "label": CLASS_NAMES,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    })

    confusion_labels = CLASS_NAMES.copy()
    if "unknown" in y_pred:
        confusion_labels.append("unknown")
    confusion_wide = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=confusion_labels),
        index=confusion_labels,
        columns=confusion_labels,
    ).rename_axis("true_label").reset_index()
    confusion_long = confusion_wide.melt(
        id_vars="true_label",
        var_name="predicted_label",
        value_name="count",
    )
    confusion_long.insert(0, "mode", mode)
    confusion_long.insert(0, "alpha", alpha)
    return summary, per_class, confusion_wide, confusion_long


# ============================================================
# MAIN LOOP
# ============================================================

df = prepare_dataset(TEST_PATH)
if MAX_ROWS is not None:
    df = df.head(MAX_ROWS).copy()

all_rows = []
all_metrics = []
all_per_class = []
all_confusion = []
baseline_cache = {}

print(f"Dataset size: {len(df)}")

for alpha in ALPHA_GRID:
    print("\n" + "#" * 120)
    print(f"ALPHA = {alpha}")
    print("#" * 120)

    alpha_rows = []
    for i, row in tqdm(df.iterrows(), total=len(df)):
        text = str(row[TEXT_COLUMN])
        true_label = str(row["true_label"])

        for run in range(N_RUNS):
            cache_key = (i, run)
            if cache_key not in baseline_cache:
                baseline_cache[cache_key] = generate_label(text, alpha=0.0)
            raw_off, predicted_off = baseline_cache[cache_key]
            raw_on, predicted_on = generate_label(text, alpha=alpha)

            print("\n" + "=" * 80)
            print(f"[{i}] alpha={alpha} run={run + 1}")
            print("TEXT:", text)
            print("TRUE:", true_label)
            print("OFF:", predicted_off, "| RAW:", raw_off)
            print("ON :", predicted_on, "| RAW:", raw_on)

            result = row.to_dict()
            result.update({
                "alpha": alpha,
                "row_id": i,
                "run": run,
                "predicted_off": predicted_off,
                "predicted_on": predicted_on,
                "raw_output_off": raw_off,
                "raw_output_on": raw_on,
                "correct_off": true_label == predicted_off,
                "correct_on": true_label == predicted_on,
            })
            alpha_rows.append(result)
            all_rows.append(result)

    alpha_df = pd.DataFrame(alpha_rows)
    summary_off, per_class_off, confusion_off, confusion_long_off = calculate_metrics(
        alpha_df, "predicted_off", "without_steering", alpha
    )
    summary_on, per_class_on, confusion_on, confusion_long_on = calculate_metrics(
        alpha_df, "predicted_on", "with_steering", alpha
    )
    summary_off["delta_macro_f1"] = 0.0
    summary_on["delta_macro_f1"] = summary_on["macro_f1"] - summary_off["macro_f1"]

    all_metrics.extend([summary_off, summary_on])
    all_per_class.extend([per_class_off, per_class_on])
    all_confusion.extend([confusion_long_off, confusion_long_on])

    print("\nMetrics without steering:", summary_off)
    print("Metrics with steering:", summary_on)
    print("\nConfusion matrix without steering:")
    print(confusion_off.to_string(index=False))
    print("\nConfusion matrix with steering:")
    print(confusion_on.to_string(index=False))


# ============================================================
# SAVE XLSX + PARQUET
# ============================================================

results_df = pd.DataFrame(all_rows)
metrics_df = pd.DataFrame(all_metrics)
per_class_df = pd.concat(all_per_class, ignore_index=True)
confusion_df = pd.concat(all_confusion, ignore_index=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
excel_results = results_df.copy()
for column in excel_results.columns:
    excel_results[column] = excel_results[column].map(excel_safe)

with pd.ExcelWriter(XLSX_OUT_PATH) as writer:
    excel_results.to_excel(writer, sheet_name="predictions", index=False)
    metrics_df.to_excel(writer, sheet_name="summary", index=False)
    per_class_df.to_excel(writer, sheet_name="per_class", index=False)
    confusion_df.to_excel(writer, sheet_name="confusion_matrix", index=False)

results_df.to_parquet(PARQUET_OUT_PATH, index=False)
metrics_df.to_parquet(METRICS_PARQUET_PATH, index=False)
per_class_df.to_parquet(PER_CLASS_PARQUET_PATH, index=False)
confusion_df.to_parquet(CONFUSION_PARQUET_PATH, index=False)

print("\nSaved XLSX results to:", XLSX_OUT_PATH)
print("Saved Parquet predictions to:", PARQUET_OUT_PATH)
print("Saved Parquet metrics to:", METRICS_PARQUET_PATH)
print("Saved Parquet per-class metrics to:", PER_CLASS_PARQUET_PATH)
print("Saved Parquet confusion matrices to:", CONFUSION_PARQUET_PATH)
